# Semantic Link Demo
This notebook explores the New York Taxi for hire vehicle data as part of the **Fabric Semantic Link Developer Experience Challenge**. 
It shows an end-2-end example of using the semantic link package for automating the semantic model generation.  

## Abstract – Automating Semantic Model Management with Semantic Link & TOM
Managing and maintaining semantic models at scale in Microsoft Fabric and Power BI introduces significant friction for analytics teams. Model logic is often fragmented across multiple tools, manual UI steps, and individual developers’ desktops, making changes hard to track, difficult to reuse, and error‑prone to roll out consistently across environments and models. This becomes even more challenging when advanced features such as calculation groups, standardized measures, or governance patterns need to be applied repeatedly.

To address this, We use the Semantic Link packages combined with the Tabular Object Model (TOM) to fully automate semantic model management from a single, version‑controlled notebook. All model logic, including metadata changes, measures, calculation groups (such as time intelligence), and standard configurations, is defined programmatically and executed in a controlled, repeatable way.

The notebook consist of several parts: 

- Part 0: Install and import essentials
- Part 1: Get Data
- Part 2: Clean Data 
- Part 3: Model Data
- Part 4: Generate Semantic Model 

The last part is the main focus of this notebook, to showcase how generation of the semantic model package works with the Tabular Object Model (TOM)

Note: The Date Dimension table has already been created up front by a different notebook.


## Problem & Solution Summary
This notebook en development is based upon the following problem and solution.

### Problem

- Semantic model changes are often manual, UI‑driven, and are hard to track
- Business logic is duplicated across models and environments
- Advanced features like calculation groups are costly to reimplement repeatedly
- Limited transparency and reusability in traditional development workflows

### Solution

- Use Semantic Link + TOM to manage semantic models programmatically, based on metadata
- Centralize all logic in a single, auditable notebook
- Automate repetitive tasks such as calculation groups and standard measures, based on datatypes
- Integrate seamlessly with DevOps for versioning, review, and deployment of the notebook
- Make fully use of the metadata of the tables


# Part 0: Install and Import essentials
Let's first start with importing the packagesand install semantic link as well.

In [ ]:
%pip install semantic-link==0.8.4 semantic-link-labs==0.8.11

In [7]:
import sempy
import logging
import pandas as pd
from typing import Optional

from pyspark.sql.functions import col, current_timestamp

import sempy.fabric as fabric
import sempy_labs._icons as icons
from sempy_labs.tom import connect_semantic_model
from sempy_labs._refresh_semantic_model import refresh_semantic_model
from sempy_labs._helper_functions import (
    resolve_lakehouse_name,
    resolve_workspace_name_and_id,
    resolve_dataset_id,
)
sempy.fabric._client._utils._init_analysis_services()
import Microsoft.AnalysisServices.Tabular as TOM
from Microsoft.AnalysisServices.Tabular import Server, Database, Model, Table, Column, Measure

from notebookutils.mssparkutils.credentials import getToken, getSecret
from notebookutils.mssparkutils.lakehouse import list as list_lakehouse

logger = logging.getLogger()
logger.setLevel(logging.INFO)
logging.debug("test")

base_url = "https://api.powerbi.com/v1.0/myorg"
token = getToken("https://analysis.windows.net/powerbi/api")


StatementMeta(, 5aece0fa-6709-4f01-899f-3cb78ad28a23, 14, Finished, Available, Finished, False)

# Part 1: Get Data

### Taxi Data
For demo purposes, we use the demo data from the New York taxi for hive vehicle. More information about this dataset can be found here:

https://learn.microsoft.com/en-us/azure/open-datasets/dataset-taxi-for-hire-vehicle?tabs=azureml-opendatasets


In [3]:
# Azure storage access info
blob_account_name = "azureopendatastorage"
blob_container_name = "nyctlc"
blob_relative_path = "fhv"
blob_sas_token = r""

# Allow SPARK to read from Blob remotely
wasbs_path = 'wasbs://%s@%s.blob.core.windows.net/%s' % (blob_container_name, blob_account_name, blob_relative_path)
spark.conf.set(
  'fs.azure.sas.%s.%s.blob.core.windows.net' % (blob_container_name, blob_account_name),
  blob_sas_token)
logging.info('Remote blob path: ' + wasbs_path)

#SPARK read parquet
df = spark.read.parquet(wasbs_path)

# Write the DataFrame to Lakehouse as a table
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("lh_bronze.dbo.nycfhv")
logging.info("Data has been written to the Lakehouse table 'lh_bronze.dbo.nycfhv'.")

StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 10, Finished, Available, Finished, False)

INFO:root:Remote blob path: wasbs://nyctlc@azureopendatastorage.blob.core.windows.net/fhv
INFO:root:Data has been written to the Lakehouse table 'lh_bronze.dbo.nycfhv'.


### Locations 
Download a mapping CSV for the locations id descriptions: 

In [4]:
# Source URL
url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# Read CSV
pdf = pd.read_csv(url)

# Convert to Spark DataFrame
df = spark.createDataFrame(pdf)

# Write to Lakehouse Bronze as Delta table
(
    df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable("lh_bronze.dbo.locations")
)
logging.info('Written locations from csv to delta table')

StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 11, Finished, Available, Finished, False)

/opt/spark/python/lib/pyspark.zip/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Cannot grow BufferHolder by size -8 because the size is negative
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
INFO:root:Written locations from csv to delta table


# Part 2: Clean Data to Silver
Clean and structure data from bronze to silver (basic principle)

In [5]:
def Bronze_to_silver(table: str):
    """Function to convert bronze table to a silver table

    Parameters:
    table (string): input table name from bronze layer.
    
    Things to evaluate in cleaning proces: 
    - Replace spaces in columns
    - add timestamp of load
    - delete duplicate records
    """
    df = spark.table(f"lh_bronze.dbo.{table}")

    for c in df.columns:
        df = df.withColumnRenamed(c, c.replace(" ", "_"))

    df = df.withColumn("_ingestion_ts", current_timestamp())
    df = df.dropDuplicates()
    logging.info(f"write table from bronze tot silver: lh_bronze.dbo.{table}")
    (
        df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(f"lh_silver.dbo.{table}")
    )

StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 12, Finished, Available, Finished, False)

Execute cleaning for all tables in bronze

In [6]:
# List all tables in the database
tables = spark.sql(f"SHOW TABLES IN lh_bronze.dbo").collect()

# list all columns from all tables
for table in tables:
    Bronze_to_silver(table['tableName'])


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 13, Finished, Available, Finished, False)

INFO:root:write table from bronze tot silver: lh_bronze.dbo.locations
INFO:root:write table from bronze tot silver: lh_bronze.dbo.nycfhv


# Part 3: Model Data for Gold
Let's model the data such that we can use it for Semantic Model generation.

Therefore we are storing Materialized Lake Views with the following metadata structure (naming conventions):

This is determined based on naming conventions:
1)  A fact starts with fact_,
2)  A dimension starts with dim_
3)  A primary key is named: {table_name}_KEY. 
4)  Columns don't have special signs, and are formatted UpperCamelCase

This applies to both facts and dimensions. This means that a fact has a primary key and foreign keys. Relationships are determined based on these foreign keys.



### DIM Doloction

Create a dim drop off location to link the drop off ID to the location data.

In [7]:
%%sql
CREATE OR REPLACE MATERIALIZED LAKE VIEW lh_gold.dbo.dim_dolocation
as
SELECT distinct
    -- Primary key
	locationId	AS dolocation_KEY,
    -- Attributes
	zone as  DropOffZone,
	borough as DropOffDistrict
FROM lh_silver.dbo.locations loc


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 14, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 2 fields>

### DIM Pulocation

Create a dim pick up location to link the pick up ID to the location data.


In [8]:
%%sql
CREATE OR REPLACE MATERIALIZED LAKE VIEW lh_gold.dbo.dim_pulocation
as
SELECT distinct
    -- Primary key
	locationId	AS pulocation_KEY,

    -- Attributes
	zone as PickUpZone,
	borough as PickUpDistrict
FROM lh_silver.dbo.locations loc


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 15, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 2 fields>

### FACT Trips

Create a fact to measure all trip durations. With primary KEY trips_KEY and foreign key's to the dimenssion. 

Note: The Date Dimension table has already been created up front by a different notebook. 

In [9]:
%%sql
CREATE OR REPLACE MATERIALIZED LAKE VIEW lh_gold.dbo.fact_trips
as
SELECT 
     -- Primary key
     CONCAT(dispatchBaseNum, '_', pickupDateTime) AS Trips_KEY,

     -- Business key
     dispatchbasenum,

     -- Foreign Keys
     COALESCE(NULLIF(puloc.locationId, ''), -1) AS pulocation_KEY,
     COALESCE(NULLIF(doloc.locationId, ''), -1) AS dolocation_KEY,
     CAST(date_format(pickupDateTime, 'yyyyMMdd') AS INT) AS date_KEY,

     -- Date & time
     date_format(pickupDateTime, 'dd-MM-yyyy') AS PickUpDate,
     date_format(pickupDateTime, 'HH:mm:ss') AS PickUpTime,
     date_format(dropOffDateTime, 'dd-MM-yyyy') AS DropOffDate,
     date_format(dropOffDateTime, 'HH:mm:ss') AS DropOffTime,

     -- Attributes
     srflag,

     -- Units & measures
     CAST((unix_timestamp(dropOffDateTime) - unix_timestamp(pickupDateTime)) / 60 AS BIGINT) AS Duration

FROM lh_silver.dbo.nycfhv AS nyc

LEFT JOIN lh_silver.dbo.locations AS puloc
ON nyc.puLocationId = puloc.locationId

LEFT JOIN lh_silver.dbo.locations AS doloc
ON nyc.doLocationId = doloc.locationId

where nyc.dropOffDateTime is not null 
and nyc.pickupDateTime is not null 
and CAST((unix_timestamp(dropOffDateTime) - unix_timestamp(pickupDateTime)) / 60 AS BIGINT) between -10000000 and 10000000

StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 16, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 2 fields>

# Part 4: Generate Semantic Model
Let's model the data such that we can use it for Semantic Model generation

In [9]:
DatasetName = "TAXI_DEMO" # define semantisch model name
lakehouse_name = 'lh_gold'

StatementMeta(, 5aece0fa-6709-4f01-899f-3cb78ad28a23, 16, Finished, Available, Finished, False)

## Get basis connections ID's of workspace

This code programmatically retrieves metadata from the current Microsoft Fabric workspace using the Fabric REST API. It identifies the active workspace, fetches all items within that workspace, and extracts the unique identifiers for a specific Lakehouse and its associated SQL Analytics Endpoint based on their display names.
The retrieved IDs can then be used for further automation scenarios such as querying metadata, managing permissions, orchestrating processes, or integrating with other Fabric APIs.

In [11]:
#Instantiate the client
client = fabric.FabricRestClient()

workspace_id = fabric.get_notebook_workspace_id()
workspaces = fabric.list_workspaces()
workspace_row = workspaces[workspaces.Id == workspace_id]

# retrieve the workspace name
workspaceName = workspace_row['Name'].values[0] 
#get a list of items from a workspace
response = client.get(f"/v1/workspaces/{workspace_id}/items")
df_items = pd.json_normalize(response.json()['value'])
lakehouse_Info= df_items[(df_items['displayName'] == lakehouse_name) & (df_items['type']== "Lakehouse")]

# retrieve the lakehouse_id
lakehouse_id = lakehouse_Info['id'].iloc[0]

SQLEndpoint_Info = df_items[(df_items['displayName'] == lakehouse_name) & (df_items['type']== "SQLEndpoint")]

# retrieve the SQLEndpoint_ID
SQLEndpoint_id = SQLEndpoint_Info['id'].iloc[0]


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 18, Finished, Available, Finished, False)

In [12]:
# Call the function

def get_sql_endpoint_str(workspaceID, lakehouseID):

    client = fabric.FabricRestClient()
    response = client.get(f"/v1/workspaces/{workspaceID}/lakehouses/{lakehouseID}")
    responseJson = response.json()
    sqlEndpoint = responseJson['properties']['sqlEndpointProperties']['connectionString']
    
    return sqlEndpoint

# retrieve SQL_ENDPOINT string value
sql_endpoint = get_sql_endpoint_str(workspace_id, lakehouse_id)


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 19, Finished, Available, Finished, False)

## Create functions

Function to create a blank (empty) semantic model

In [13]:
def create_blank_semantic_model(
    dataset: str,
    compatibility_level: int = 1604,
    workspace: Optional[str] = None,
    overwrite: bool = True,
):
    """
    Creates a new blank semantic model (no tables/columns etc.).

    Parameters
    ----------
    dataset : str
        Name of the semantic model.
    compatibility_level : int, default=1604
        The compatibility level of the semantic model.
    workspace : str, default=None
        The Fabric workspace name.
        Defaults to None which resolves to the workspace of the attached lakehouse
        or if no lakehouse attached, resolves to the workspace of the notebook.
    overwrite : bool, default=False
        If set to True, overwrites the existing semantic model in the workspace if it exists.
    """

    workspace = fabric.resolve_workspace_name(workspace)
    dfD = fabric.list_datasets(workspace=workspace, mode="rest")
    dfD_filt = dfD[dfD["Dataset Name"] == dataset]

    if len(dfD_filt) > 0 and not overwrite:
        raise ValueError(
            f"{icons.warning} The '{dataset}' semantic model already exists within the '{workspace}' workspace. The 'overwrite' parameter is set to False so the blank new semantic model was not created."
        )

    min_compat = 1500
    if compatibility_level < min_compat:
        raise ValueError(
            f"{icons.red_dot} Compatiblity level must be at least {min_compat}."
        )

    tmsl = f"""
    {{
        "createOrReplace": {{
        "object": {{
            "database": '{dataset}'
        }},
        "database": {{
            "name": '{dataset}',
            "compatibilityLevel": {compatibility_level},
            "model": {{
            "culture": "en-US",
            "defaultPowerBIDataSourceVersion": "powerBI_V3"
            }}
        }}
        }}
    }}
    """

    fabric.execute_tmsl(script=tmsl, workspace=workspace)

    return logging.info(
        f"{icons.green_dot} The '{dataset}' semantic model was created within the '{workspace}' workspace."
    )



StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 20, Finished, Available, Finished, False)

Function to map data types from delta table to the semantic model

In [14]:
# Mapping function
def map_data_type(data_type):
    """"Function to map data types""" 
    mapping = {
        'bigint': 'Int64',
        'int': 'Int64',
        'string': 'String',
        'float': 'Decimal',
        'double': 'Decimal',
        'decimal(18,4)': 'Decimal',
        'datetime': 'DateTime',
        'timestamp': 'DateTime',
        'date': 'DateTime',
        'boolean': 'Boolean',
        'void': 'String'
    }
    return mapping.get(data_type, 'Other')  # Default to 'Other' if no match


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 21, Finished, Available, Finished, False)

## Retrieve metadata data from datawarahouse 

This code scans all tables in the specified Lakehouse database and extracts column metadata for dimension and fact tables. For each relevant table, it retrieves schema details, enriches them with model‑related attributes such as data type mappings and summarization behavior, and combines the results into a single Pandas DataFrame. The final output provides a consolidated view of column metadata, ready for further semantic modeling or automation.

In [15]:
# Initialize an empty Pandas DataFrame
dataframes = []

# List all tables in the database
tables = spark.sql(f"SHOW TABLES IN {lakehouse_name}.dbo").collect()
# Loop through each table and describe it

# list all columns from all tables
for table in tables:
    table_name = table['tableName']
    if table_name.startswith("dim_") or table_name.startswith("fact"):
        logging.info(f"Processing table {table_name}...")
        
        # Get the columns of the table
        df = spark.sql(f"DESCRIBE {lakehouse_name}.dbo.{table_name}")

        # Convert Spark DataFrame to Pandas DataFrame
        pandas_df = df.toPandas()
        # Add the table_name column
        pandas_df['table_name'] = table_name
        # Add a new column based on the datatype
        pandas_df['is_string'] = pandas_df['data_type'].apply(lambda x: 'string' in x.lower())
        pandas_df['summarize_by'] = pandas_df['is_string'].apply(lambda x: None if x else 'Default')
        pandas_df['datatypemodel'] = pandas_df['data_type'].apply(map_data_type)
        # Remove the 'comment' column
        pandas_df = pandas_df.drop(columns=['comment'])
        # Append to the list
        dataframes.append(pandas_df)
    else:
        logging.info(f"Skip {table_name} as it is not a dim or fact table")

# Concatenate all DataFrames in the list into a single DataFrame
final_df = pd.concat(dataframes, ignore_index=True)


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 22, Finished, Available, Finished, False)

INFO:root:Processing table dim_date...
INFO:root:Processing table dim_dolocation...
INFO:root:Processing table dim_pulocation...
INFO:root:Processing table fact_trips...
INFO:root:Skip sys_dq_metrics as it is not a dim or fact table


## Metrics DF
Get, based on the metadata data types all INT and Decimal columns. 
These columns will be defined as the columns that get a standard measure in the model.

In [16]:
df = final_df[(final_df['datatypemodel']==('Int64')) | (final_df['datatypemodel']==('Decimal') )]
# only for fact tables table and remove _KEY columns:
Measure_set = df.copy()
Measure_df = Measure_set[(Measure_set['table_name'].str.contains('fact_')) & (~Measure_set['col_name'].str.contains('_KEY'))]

logging.info(Measure_df)

StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 23, Finished, Available, Finished, False)

INFO:root:    col_name data_type  table_name  is_string summarize_by datatypemodel
48  Duration    bigint  fact_trips      False      Default         Int64


## Relationships DF
This code identifies key columns by selecting all columns that follow a _KEY naming convention. It then derives the related dimension and fact relationships by parsing table and column names, validates matching dimension keys, excludes special cases, and produces a filtered list of valid key relationships for further modeling or relationship creation.

In [17]:
#get all key columns
key_rows_all = final_df[(final_df['col_name'].str.contains('_KEY'))]

# split columns for names
split_columns_table= key_rows_all['table_name'].str.split('_', expand=True)
key_rows_all = key_rows_all.copy()
key_rows_all[['fact_or_dim', 'new_table_name']] =split_columns_table

split_columns_key= key_rows_all['col_name'].str.split('_', expand=True)
key_rows_all = key_rows_all.copy()
key_rows_all[['dim_table_name', 'KEY']] =split_columns_key

key_rows_all['dim_table_name'] = key_rows_all['dim_table_name'].str.lower()

key_rows_key =key_rows_all[key_rows_all['new_table_name'] == key_rows_all['dim_table_name']]
key_rows_key =key_rows_key[key_rows_key['table_name'] != 'dim_date']
# nodig voor key columns
logging.info(key_rows_key)

StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 24, Finished, Available, Finished, False)

INFO:root:          col_name data_type      table_name  is_string summarize_by  \
32  dolocation_KEY    bigint  dim_dolocation      False      Default   
35  pulocation_KEY    bigint  dim_pulocation      False      Default   
38       Trips_KEY    string      fact_trips       True          NaN   

   datatypemodel fact_or_dim new_table_name dim_table_name  KEY  
32         Int64         dim     dolocation     dolocation  KEY  
35         Int64         dim     pulocation     pulocation  KEY  
38        String        fact          trips          trips  KEY  


Create the list of relationships based on the key rows.

In [18]:
key_rows = key_rows_all[(key_rows_all['table_name'].str.contains('^fact_'))]

# # retreive rows for relations
filtered_df = key_rows[key_rows['new_table_name'] != key_rows['dim_table_name']]


filtered_df= filtered_df.copy()
filtered_df['dim_table_name'] = 'dim_'+filtered_df['dim_table_name']
# define the right name for usage
filtered_df.rename(columns={'table_name': 'fact_table_name'}, inplace=True)

Relationship_df = filtered_df

logging.info(Relationship_df)

StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 25, Finished, Available, Finished, False)

INFO:root:          col_name data_type fact_table_name  is_string summarize_by  \
40  pulocation_KEY    bigint      fact_trips      False      Default   
41  dolocation_KEY    bigint      fact_trips      False      Default   
42        date_KEY       int      fact_trips      False      Default   

   datatypemodel fact_or_dim new_table_name  dim_table_name  KEY  
40         Int64        fact          trips  dim_pulocation  KEY  
41         Int64        fact          trips  dim_dolocation  KEY  
42         Int64        fact          trips        dim_date  KEY  


## Create model, connections, tables and columns

This code programmatically creates and configures a semantic model using Semantic Link and TOM. It initializes a blank model, connects it to a SQL Analytics Endpoint, adds tables in Direct Lake mode, and dynamically generates columns based on metadata. 

Key columns are marked and hidden, sorting behavior is applied where needed, and the model is prepared for refresh with standardized structure and semantics.


In [19]:
# step 1 Create model
create_blank_semantic_model(DatasetName)

# step 2 Create database query
DatabaseQuery = f"""let database = Sql.Database(\"{sql_endpoint}\", \"{SQLEndpoint_id}\")in database""" 
       
with connect_semantic_model(dataset=DatasetName, readonly=False, workspace=workspaceName) as tom:
    # add query
    tom.add_expression("DatabaseQuery", DatabaseQuery)
    # add tables from layer
    for table in tables:
        logging.info(
        f"{icons.green_dot} The '{table.tableName}' table is added to the dataset '{DatasetName}'."
        )
        tom.add_table(name=table.tableName)
        tom.add_entity_partition(table_name=table.tableName, entity_name=table.tableName) 

# Step 3
# connect to model and apply updates for Expression, and table and partition in directLake mode
with connect_semantic_model(dataset=DatasetName, readonly=False, workspace=workspaceName) as tom:
    for index, row in final_df.iterrows():
        logging.info(
        f"{icons.green_dot} '{row['col_name']}' is added to the '{row['table_name']}' table."
            )
        if row['col_name'].endswith('_KEY'):
            logging.info(f"Adding {row['col_name']}")
            tom.add_data_column(table_name=row['table_name'], column_name=row['col_name'], source_column=row['col_name'], data_type=row['datatypemodel'], summarize_by="Default",hidden=True)
        else:
            summarize = row['summarize_by']
            if not isinstance(summarize, str) or summarize.strip() == "":
                summarize = "Default"
            tom.add_data_column(table_name=row['table_name'], column_name=row['col_name'], source_column=row['col_name'], data_type=row['datatypemodel'], summarize_by=summarize)
            if row['col_name'] == 'MaandNaam':
                tom.set_sort_by_column(table_name=row['table_name'], column_name=row['col_name'],sort_by_column="Maand")
            if row['col_name'] == 'WeekdagOmschrijving':
                tom.set_sort_by_column(table_name=row['table_name'], column_name=row['col_name'],sort_by_column="DagVanDeWeek")

    for index, row in key_rows_key.iterrows():
        logging.info(f"'{row['col_name']}' will be signed as key column in table")
        tom.update_column(table_name=row['table_name'], column_name=row['col_name'], key=True, hidden=True)



StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 26, Finished, Available, Finished, False)

INFO:root:🟢 The 'TAXI_DEMO' semantic model was created within the 'WS_Benito' workspace.
INFO:root:🟢 The 'dim_date' table is added to the dataset 'TAXI_DEMO'.
INFO:root:🟢 The 'dim_dolocation' table is added to the dataset 'TAXI_DEMO'.
INFO:root:🟢 The 'dim_pulocation' table is added to the dataset 'TAXI_DEMO'.
INFO:root:🟢 The 'fact_trips' table is added to the dataset 'TAXI_DEMO'.
INFO:root:🟢 The 'sys_dq_metrics' table is added to the dataset 'TAXI_DEMO'.
INFO:root:🟢 'date_KEY' is added to the 'dim_date' table.
INFO:root:Adding date_KEY
INFO:root:🟢 'date' is added to the 'dim_date' table.
INFO:root:🟢 'datetime' is added to the 'dim_date' table.
INFO:root:🟢 'year' is added to the 'dim_date' table.
INFO:root:🟢 'year_description' is added to the 'dim_date' table.
INFO:root:🟢 'tertiary' is added to the 'dim_date' table.
INFO:root:🟢 'quarter' is added to the 'dim_date' table.
INFO:root:🟢 'quarter_code' is added to the 'dim_date' table.
INFO:root:🟢 'quarter_description' is added to the 'dim_d

## Add Date Hierachies

Add three default date hierarchies to the deate dimension:
1. Year - Month - Date
2. Year - Quarter - Month - Date
3. Year - Week - Date

In [20]:
with connect_semantic_model(dataset=DatasetName, readonly=False, workspace=workspaceName) as tom:
    logging.info(
    f"{icons.green_dot} adding hierarchies to date dimension"
        )
    tom.add_hierarchy(table_name='dim_date', hierarchy_name='yyyy-mm-dd', columns=['Year', 'Month', 'Date'], levels=['Year', 'Month', 'Date'])
    tom.add_hierarchy(table_name='dim_date', hierarchy_name='yyyy-qq-mm-dd', columns=['Year','Quarter', 'Month', 'Date'], levels=['Year','Quarter', 'Month', 'Date'])
    tom.add_hierarchy(table_name='dim_date', hierarchy_name='yyyy-wk-dd', columns=['Year', 'Week', 'Date'], levels=['Year', 'Week', 'Date'])

StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 27, Finished, Available, Finished, False)

INFO:root:🟢 adding hierarchies to date dimension


## Add Relationships

Add relationships to the model based on metadata set: Relationship_df


In [21]:
with connect_semantic_model(dataset=DatasetName, readonly=False, workspace=workspaceName) as tom:
    for index, row in Relationship_df.iterrows():
        logging.info(
        f"{icons.green_dot} relationship '{row['fact_table_name']}'to '{row['dim_table_name']}' will be added."
            )
        tom.add_relationship(
            from_table=row['fact_table_name'], from_column=row['col_name'],
            to_table=row['dim_table_name'], to_column= row['col_name'],
            from_cardinality='Many', to_cardinality='One')


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 28, Finished, Available, Finished, False)

INFO:root:🟢 relationship 'fact_trips'to 'dim_pulocation' will be added.
INFO:root:🟢 relationship 'fact_trips'to 'dim_dolocation' will be added.
INFO:root:🟢 relationship 'fact_trips'to 'dim_date' will be added.


## Add Measures to the model

Add the defaul measure(s) to the semantic model, based on the metadata set: Measure_df

In [22]:
# add default SUM for these columns
with connect_semantic_model(dataset=DatasetName, readonly=False, workspace=workspaceName) as tom:
    for index, row in Measure_df.iterrows():
        DAX_EXPRESSION = f"SUM('{row['table_name']}'[{row['col_name']}])"
        tom.add_measure(table_name=row['table_name'], measure_name=f"SUM_{row['col_name']}", expression=DAX_EXPRESSION)

StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 29, Finished, Available, Finished, False)

### Customize metrics

This section can be appende per model. 
You can add you own DAX calculation to add to the model (and respective table)
As an example and AVERAGE_DURATION measure is added.

In [23]:
# this section gives the possibility to add custom metrics to each column.
# Please create a valid DAX_EXPRESSION to add to the Column
TABLE_NAME = 'fact_trips'
DAX_EXPRESSION = """AVERAGE(fact_trips[Duration])"""
MEASURE_NAME = 'AVERAGE_DURATION'

with connect_semantic_model(dataset=DatasetName, readonly=False, workspace=workspaceName) as tom:
    tom.add_measure(table_name=TABLE_NAME, measure_name=MEASURE_NAME, expression=DAX_EXPRESSION, format_string= "0.00")


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 30, Finished, Available, Finished, False)

## Time-Intelligence
Add Calculation group for Time-Intelligence: Standaard

### DAX Aggregation

This code defines a set of reusable DAX expressions for time intelligence calculations.

The expressions calculate Year‑to‑Date (YTD) and Quarter‑to‑Date (QTD) values based on the latest available period in the date dimension, as well as Last Year equivalents for MTD, QTD, and YTD. By using SELECTEDMEASURE(), these expressions are designed to be applied generically, typically within calculation groups, so they automatically work for any measure in the semantic model while ensuring correct filtering and consistent time‑based behavior.

In [24]:
# dates TimeIntelligenceAggregation

DAX_YTD ="""VAR LastMonthAvailable = MAX ( 'dim_date'[year_month_order] )
VAR LastYearAvailable = MAX ( 'dim_date'[year] )
VAR Result =
    CALCULATE (
        SELECTEDMEASURE(),
        REMOVEFILTERS ( 'dim_date' ),
        'dim_date'[year_month_order] <= LastMonthAvailable,
        'dim_date'[year] = LastYearAvailable
    )
RETURN
    Result
"""

DAX_QTD = """
VAR LastMonthAvailable = MAX ( 'dim_date'[year_month_order] )
VAR LastYearQuarterAvailable = MAX ( 'dim_date'[year_quarter_order] )
VAR Result =
    CALCULATE (
        SELECTEDMEASURE(),
        REMOVEFILTERS ( 'dim_date' ),
        'dim_date'[year_month_order] <= LastMonthAvailable,
        'dim_date'[year_quarter_order] = LastYearQuarterAvailable
    )
RETURN
    Result
"""

DAX_LYMTD ="""CALCULATE(SELECTEDMEASURE(), 'TimeIntelligenceAggregation'[Aggregation] = "MTD", SAMEPERIODLASTYEAR('dim_date'[date]), REMOVEFILTERS ( 'dim_date' ))"""
DAX_LYQTD ="""CALCULATE(SELECTEDMEASURE(), 'TimeIntelligenceAggregation'[Aggregation] = "QTD", SAMEPERIODLASTYEAR('dim_date'[date]), REMOVEFILTERS ( 'dim_date' ))"""
DAX_LYYTD ="""CALCULATE(SELECTEDMEASURE(), 'TimeIntelligenceAggregation'[Aggregation] = "YTD", SAMEPERIODLASTYEAR('dim_date'[date]), REMOVEFILTERS ( 'dim_date' ))"""


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 31, Finished, Available, Finished, False)

### DAX Comparison

These DAX expressions define reusable time‑based comparison calculations for semantic models. 

They cover Last Year (LY), Year‑over‑Year (YOY) and Month‑over‑Month (MOM) differences and percentages, as well as Last Month (LM), Last Quarter (LQ), Quarter‑over‑Quarter (QOQ), and Moving Annual Total (MAT). By using SELECTEDMEASURE(), the logic is generic and typically used in calculation groups, ensuring consistent and scalable time intelligence across all measures.

In [25]:

DAX_LY = """
VAR CurrentYearNumber = SELECTEDVALUE ( 'dim_date'[year] )
VAR PreviousYearNumber = CurrentYearNumber - 1
VAR Result =
    CALCULATE (
        SELECTEDMEASURE(),
        REMOVEFILTERS ( 'dim_date' ),
        'dim_date'[year] = PreviousYearNumber,
        VALUES ( 'dim_date'[month] )
    )
RETURN
    Result
"""

DAX_YOY= """
VAR ValueCurrentPeriod = SELECTEDMEASURE()
VAR ValuePreviousPeriod = CALCULATE(SELECTEDMEASURE(), 'TimeIntelligenceComparison'[Comparison] = "LY")
VAR Result =
    IF (
        NOT ISBLANK ( ValueCurrentPeriod ) && NOT ISBLANK ( ValuePreviousPeriod ),
        ValueCurrentPeriod - ValuePreviousPeriod
    )
RETURN
    Result
"""

DAX_YOYperc ="""
DIVIDE(
    CALCULATE(
        SELECTEDMEASURE(),
        'TimeIntelligenceComparison'[Comparison] = "YOY"
    ),
    CALCULATE(
        SELECTEDMEASURE(),
        'TimeIntelligenceComparison'[Comparison] = "LY"
    )
)
"""

DAX_LM = """CALCULATE (SELECTEDMEASURE(), PREVIOUSMONTH('dim_date'[date]), REMOVEFILTERS ( 'dim_date' ))"""

DAX_MOM= """
VAR ValueCurrentPeriod = SELECTEDMEASURE()
VAR ValuePreviousPeriod = CALCULATE(SELECTEDMEASURE(), 'TimeIntelligenceComparison'[Comparison] = "LM")
VAR Result =
    IF (
        NOT ISBLANK ( ValueCurrentPeriod ) && NOT ISBLANK ( ValuePreviousPeriod ),
        ValueCurrentPeriod - ValuePreviousPeriod
    )
RETURN
    Result

"""
DAX_MOMperc ="""
DIVIDE(
    CALCULATE(
        SELECTEDMEASURE(),
        'TimeIntelligenceComparison'[Comparison] = "MOM"
    ),
    CALCULATE(
        SELECTEDMEASURE(),
        'TimeIntelligenceComparison'[Comparison] = "LM"
    )
)
"""

DAX_LQ = """CALCULATE (SELECTEDMEASURE(), PREVIOUSQUARTER('dim_date'[date]), REMOVEFILTERS ( 'dim_date' ))"""

DAX_QOQ= """
VAR ValueCurrentPeriod = SELECTEDMEASURE()
VAR ValuePreviousPeriod = CALCULATE(SELECTEDMEASURE(), 'TimeIntelligenceComparison'[Comparison] = "LQ")
VAR Result =
    IF (
        NOT ISBLANK ( ValueCurrentPeriod ) && NOT ISBLANK ( ValuePreviousPeriod ),
        ValueCurrentPeriod - ValuePreviousPeriod
    )
RETURN
    Result
"""
DAX_QOQperc ="""
DIVIDE(
    CALCULATE(
        SELECTEDMEASURE(),
        'TimeIntelligenceComparison'[Comparison] = "QOQ"
    ),
    CALCULATE(
        SELECTEDMEASURE(),
        'TimeIntelligenceComparison'[Comparison] = "LQ"
    )
)
"""


DAX_MAT = """
VAR MonthsInRange = 12
VAR LastMonthRange = MAX ( 'dim_date'[year_month_order] )
VAR FirstMonthRange = LastMonthRange - MonthsInRange + 1
VAR Result =
    CALCULATE (
        SELECTEDMEASURE(),
        REMOVEFILTERS ( 'dim_date' ),
        'dim_date'[year_month_order] >= FirstMonthRange
            && 'dim_date'[year_month_order] <= LastMonthRange
    )
RETURN
    Result
"""


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 32, Finished, Available, Finished, False)

### Add to model
Both DAX Aggregation and DAX Comparison metrics will be added to the model below as calculation groups.

In [26]:
# dit werkt het beste!
with connect_semantic_model(dataset=DatasetName, readonly=False, workspace=workspaceName) as tom:
    tom.add_calculation_group(name='TimeIntelligenceAggregation', precedence=10)
    for t in tom.model.Tables:
        if t.Name=='TimeIntelligenceAggregation':
            for c in t.Columns:
                c.Name = c.Name.replace('Name', 'Aggregation')

    tom.add_calculation_item(table_name='TimeIntelligenceAggregation', calculation_item_name='Current', expression="SELECTEDMEASURE()")
    tom.add_calculation_item(table_name='TimeIntelligenceAggregation', calculation_item_name='MTD', expression="TOTALMTD(SELECTEDMEASURE(), 'dim_date'[date])")
    tom.add_calculation_item(table_name='TimeIntelligenceAggregation', calculation_item_name='QTD', expression=DAX_QTD)
    tom.add_calculation_item(table_name='TimeIntelligenceAggregation', calculation_item_name='YTD', expression=DAX_YTD)

    tom.add_calculation_item(table_name='TimeIntelligenceAggregation', calculation_item_name='LY MTD', expression=DAX_LYMTD)
    tom.add_calculation_item(table_name='TimeIntelligenceAggregation', calculation_item_name='LY QTD', expression=DAX_LYQTD)
    tom.add_calculation_item(table_name='TimeIntelligenceAggregation', calculation_item_name='LY YTD', expression=DAX_LYYTD)

with connect_semantic_model(dataset=DatasetName, readonly=False, workspace=workspaceName) as tom:
    tom.add_calculation_group(name='TimeIntelligenceComparison', precedence=20)
    for t in tom.model.Tables:
        if t.Name=='TimeIntelligenceComparison':
            for c in t.Columns:
                c.Name = c.Name.replace('Name', 'Comparison')

    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='Current', expression="SELECTEDMEASURE()",ordinal=1)
    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='LY', expression=DAX_LY, ordinal=2)
    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='YOY', expression=DAX_YOY, ordinal=3)
    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='YOY%', expression=DAX_YOYperc, ordinal=4)

    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='LM', expression=DAX_LM, ordinal=5)
    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='MOM', expression=DAX_MOM, ordinal=6)
    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='MOM%', expression=DAX_MOMperc, ordinal=7)

    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='LQ', expression=DAX_LQ, ordinal=8)
    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='QOQ', expression=DAX_QOQ, ordinal=9)
    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='QOQ%', expression=DAX_QOQperc, ordinal=10)

    tom.add_calculation_item(table_name='TimeIntelligenceComparison', calculation_item_name='MAT', expression=DAX_MAT, ordinal=11)  


StatementMeta(, a51d1e7f-6955-4950-b4a3-e970fda47639, 33, Finished, Available, Finished, False)

## Refresh Model

Refresh the created model!

In [10]:
refresh_semantic_model(dataset=DatasetName)
logging.info("Modelling done and it's ready to be used for reporting and analyses")

StatementMeta(, 5aece0fa-6709-4f01-899f-3cb78ad28a23, 17, Finished, Available, Finished, False)

⌛ Refresh of the 'TAXI_DEMO' semantic model within the 'WS_Benito' workspace is in progress...
🟢 Refresh 'full' of the 'TAXI_DEMO' semantic model within the 'WS_Benito' workspace is complete.


# Summary & Conclusion
This notebook demonstrates a complete, automated workflow for semantic model management using Microsoft Fabric’s Semantic Link and TOM APIs. It covers data ingestion, cleaning, modeling, and programmatic semantic model generation, including advanced features like calculation groups and custom measures. 

**Key Points:**
- All steps are automated and reproducible, reducing manual effort and errors.
- The notebook is well-documented, making logic and configuration transparent and easy to review.
- The approach is reusable and adaptable for different models and environments.
- Advanced modeling features (relationships, hierarchies, time intelligence) are implemented programmatically.

**Conclusion:**
This solution provides a modern, DevOps-ready approach to semantic model development. It improves developer experience, ensures consistency, and enables rapid, scalable semantic model management in Microsoft Fabric.